# RQ1 -- Solver comparison

*How do Forward Euler, Heun, and RK4 differ in accuracy, observed convergence order, and numerical stability when solving the SIR system?*

This notebook narrates the results already computed by `experiments/E01_convergence.py`, `E02_invariants.py`, `E03_stability.py`, and `E04_structural.py`. Run `make experiments` (or `make experiment ID=E01` etc.) before executing these cells.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve() / "scripts"))
from plot_style import apply_style, SOLVER_COLOR, COMPARTMENT_COLOR
apply_style()

RESULTS = Path("..") / "results"

def load(exp_id):
    return json.loads((RESULTS / f"{exp_id}.json").read_text())


## Convergence order

Each solver's error against the semi-analytic gold standard (`sirlab.reference`), as a function of step size h. The fitted log-log slope is the *observed* convergence order p&#770; -- it should match the theoretical order (1, 2, 4, 1, 2 for Euler / Heun / RK4 / backward Euler / trapezoidal).

In [ ]:
d = load("E01")
regime = d["regimes"][0]
fig, ax = plt.subplots(figsize=(6, 4.5))
for solver, res in regime["solvers"].items():
    ax.loglog(res["h"], res["error_final_state"], "o-", color=SOLVER_COLOR.get(solver, "gray"),
               label=f"{solver} (p̂={res['observed_order']:.2f}, theory={res['theoretical_order']})")
ax.set_xlabel("step size h"); ax.set_ylabel(f"error at t={d['t_final']}")
ax.set_title(f"Convergence -- {regime['name']}")
ax.legend()
plt.show()


## The mass-conservation-is-uninformative finding

Because the SIR right-hand side sums to zero identically, every explicit Runge-Kutta method conserves S+I+R to roundoff **at any step size** -- including a solve that is otherwise catastrophically wrong. The phase invariant Q(t) = ln S + (beta/gamma N) R drifts at the solver's true convergence order instead, and is the diagnostic that actually discriminates solver quality.

In [ ]:
d2 = load("E02")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for solver, per_h in d2["results"].items():
    for h_str, entry in per_h.items():
        axes[0].semilogy(entry["t"], np.maximum(entry["mass_error"], 1e-18), color=SOLVER_COLOR.get(solver, "gray"), alpha=0.6)
        axes[1].semilogy(entry["t"], np.maximum(entry["phase_invariant_drift"], 1e-18), color=SOLVER_COLOR.get(solver, "gray"), alpha=0.6)
axes[0].set_title("Mass error (uninformative)"); axes[1].set_title("Phase-invariant drift (discriminating)")
plt.show()


## Stability frontier and structural accuracy

See `E03` (empirical stability boundary in (β, γ, h)) and `E04` (peak height / peak time / final-size errors) for the remaining RQ1 evidence.

In [ ]:
d3 = load("E03")
fig, axes = plt.subplots(1, len(d3["solvers"]), figsize=(4 * len(d3["solvers"]), 4))
for ax, (solver, res) in zip(np.atleast_1d(axes), d3["solvers"].items()):
    im = ax.pcolormesh(d3["beta_grid"], d3["gamma_grid"], np.array(res["first_failure_h"]).T, shading="auto")
    ax.set_title(f"{solver}: largest stable h"); ax.set_xlabel("β"); ax.set_ylabel("γ")
    fig.colorbar(im, ax=ax)
plt.show()
